In [1]:
# ===== Cell 1: imports and local paths =====
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ===== Cell 2: local save path =====
save_dir = os.path.join('.', 'outputs', 'models')
os.makedirs(save_dir, exist_ok=True)

# ===== Cell 3: device =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# ===== Cell 4: data =====
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Keep only digits 2 and 6
train_mask = (train_dataset.targets == 2) | (train_dataset.targets == 6)
test_mask = (test_dataset.targets == 2) | (test_dataset.targets == 6)

train_indices = torch.where(train_mask)[0]
test_indices = torch.where(test_mask)[0]

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

print(f"Train samples (2 or 6): {len(train_subset)}")
print(f"Test samples (2 or 6): {len(test_subset)}")

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=128, shuffle=False)

# ===== Cell 5: config =====
latent_dims = [16, 32, 64, 80]
epochs_vae = 6
epochs_clf = 2
lr = 1e-3

# ===== Cell 6: model definitions (aligned with vae.ipynb) =====
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=80):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x_hat = self.decoder(z)
        return x_hat.view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ===== Cell 7: training functions =====
def vae_loss(recon_x, x, mu, logvar):
    bce = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld


def train_vae(
    vae: nn.Module,
    train_loader: DataLoader,
    device: str,
    epochs: int = 6,
    lr: float = 1e-3,
):
    optimizer = optim.Adam(vae.parameters(), lr=lr)
    vae.train()

    final_avg_loss = None
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon_x, mu, logvar = vae(x)
            loss = vae_loss(recon_x, x, mu, logvar)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        final_avg_loss = running_loss / len(train_loader.dataset)
        print(f"VAE epoch {epoch}/{epochs}, loss={final_avg_loss:.6f}")

    return float(final_avg_loss)


def train_classifier(
    clf: nn.Module,
    train_loader: DataLoader,
    device: str,
    epochs: int = 2,
    lr: float = 1e-3
):
    optimizer = optim.Adam(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    clf.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            logits = clf(x)
            loss = loss_fn(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        avg_loss = running_loss / len(train_loader)
        acc = correct / total
        print(f"CLF epoch {epoch + 1}/{epochs}, loss = {avg_loss:.6f}, acc = {acc:.4f}")


# ===== Cell 8: evaluation function =====
@torch.no_grad()
def evaluate_classifier(clf: nn.Module, data_loader: DataLoader, device: str):
    clf.eval()

    correct = 0
    total = 0

    for x, y in data_loader:
        x = x.to(device)
        y = y.to(device)

        logits = clf(x)
        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    print(f"Test accuracy = {acc:.4f}")
    return float(acc)


# ===== Cell 9: main =====
def main():
    # Train one shared classifier for reward evaluation in RL notebook.
    clf = Classifier().to(device)
    train_classifier(
        clf=clf,
        train_loader=train_loader,
        device=device,
        epochs=epochs_clf,
        lr=lr
    )
    test_acc = evaluate_classifier(
        clf=clf,
        data_loader=test_loader,
        device=device
    )
    torch.save(clf.state_dict(), os.path.join(save_dir, "classifier.pt"))

    rows = []
    for latent_dim in latent_dims:
        print(f"\n=== Training VAE latent_dim={latent_dim} ===")
        vae = VAE(latent_dim=latent_dim).to(device)
        final_loss = train_vae(
            vae=vae,
            train_loader=train_loader,
            device=device,
            epochs=epochs_vae,
            lr=lr,
        )

        vae_name = f"vae_{latent_dim}.pt"
        torch.save(vae.state_dict(), os.path.join(save_dir, vae_name))
        rows.append({
            "latent_dim": latent_dim,
            "final_vae_loss": final_loss,
            "classifier_test_acc": test_acc,
            "vae_checkpoint": vae_name,
        })

    summary_df = pd.DataFrame(rows).sort_values("latent_dim")
    summary_csv = os.path.join(save_dir, "vae_dim_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("\nSaved checkpoints and summary:")
    print(summary_df)
    print(f"Summary CSV: {summary_csv}")
    print(os.listdir(save_dir))


# ===== Cell 10: run =====
if __name__ == "__main__":
    main()

cpu
Train samples (2 or 6): 11876
Test samples (2 or 6): 1990
CLF epoch 1/2, loss = 0.270919, acc = 0.9525
CLF epoch 2/2, loss = 0.046863, acc = 0.9858
Test accuracy = 0.9899

=== Training VAE latent_dim=16 ===
VAE epoch 1/6, loss=231.510105
VAE epoch 2/6, loss=164.372009
VAE epoch 3/6, loss=145.794955
VAE epoch 4/6, loss=135.140529
VAE epoch 5/6, loss=129.271941
VAE epoch 6/6, loss=125.136795

=== Training VAE latent_dim=32 ===
VAE epoch 1/6, loss=231.803235
VAE epoch 2/6, loss=167.928093
VAE epoch 3/6, loss=151.285719
VAE epoch 4/6, loss=140.418707
VAE epoch 5/6, loss=133.839532
VAE epoch 6/6, loss=129.178105

=== Training VAE latent_dim=64 ===
VAE epoch 1/6, loss=237.089448
VAE epoch 2/6, loss=173.871538
VAE epoch 3/6, loss=157.023704
VAE epoch 4/6, loss=146.439047
VAE epoch 5/6, loss=140.200684
VAE epoch 6/6, loss=135.718799

=== Training VAE latent_dim=80 ===
VAE epoch 1/6, loss=233.767290
VAE epoch 2/6, loss=172.000926
VAE epoch 3/6, loss=156.634071
VAE epoch 4/6, loss=147.404668

In [ ]:
!jupyter nbconvert --to script ./haoting_results/models.ipynb

[NbConvertApp] Converting notebook /content/drive/MyDrive/6501-009/GP/models.ipynb to script
[NbConvertApp] Writing 5414 bytes to /content/drive/MyDrive/6501-009/GP/models.txt
